# Laboratorio de clasificación 3

|                |   |
:----------------|---|
| **Nombre**     | Fernando Emmanuel Gonzalez Garcia   |
| **Fecha**      | 18-oct-26  |
| **Expediente** |  748126 | 

Una regresión logística simple toma **P** predictores y usa una función no lineal para convertir la respuesta binaria **Y** en probabilidades entre $0$ y $1$.

La visualización de este escenario es puntos separados por una línea recta.

<center><img src="https://wirelesspi.com/wp-content/uploads/2022/08/figure-logistic-regression-decision-boundary-2-570x550.png"></center>

Para casos donde los datos no son linealmente separables usamos el truco del kernel para mapear los datos a un espacio de Hilbert.

Imagina un escenario donde tienes dos predictores $x_1, x_2$ y la salida de estos predictores se comporta de esta manera:

| **$x_1$** | **$x_2$** | Salida |
:-----------|-----------|--------|
| 0         | 0         | 0      |
| 0         | 1         | 1      |
| 1         | 0         | 1      |
| 1         | 1         | 0      |


Imagínalo como el interruptos de luz de un foco en una escalera. Se puede prender y apagar desde arriba y desde abajo.

<center><img src="https://www.researchgate.net/profile/Brian-Spears/publication/322048911/figure/fig4/AS:667820685807618@1536232267250/The-exclusive-or-XOR-function-is-a-nonlinear-function-that-returns-0-when-its-two.png"></center>

¿Cómo podemos separar estos datos usando sólo regresión logística? Sin importar cómo usemos una sola línea, nunca se van a poder separar todos los casos, por lo que tendríamos situaciones donde siempre habría error. 

Solución: ¿Por qué nos estamos limitando a una sola línea?

<center><img src="https://pabloinsente.github.io/assets/post-5/XOR-linear.png"></center>

La regresión logística sola es un método de aprendizaje estadístico convexo. Combinar dos regresiones logísticas para clasificar datos no linealmente separables convierte el problema a uno no convexo. ¿Por qué?

- * Una regresión logística sola tiene una función de error convexa (un solo fondo en forma de tazón), por lo que siempre encuentra el punto óptimo. Al juntar dos o más regresiones (hacer una red neuronal), la salida de una entra a la otra de forma no lineal; esto deforma la función de error, creando muchos valles, pozos falsos (mínimos locales) y zonas planas, volviendo el problema no convexo. visualmente seria algo como porque esta afuera del rango de valores permitidos y al momento de graficar dos puntos de saldria del area permitida por asi decirlo dentro de los valores separados **

¿Qué significa que una función sea convexa?

- **Significa que si tomas dos puntos cualesquiera sobre la curva y trazas una línea recta entre ellos, esa línea siempre queda por encima de la función (como la forma de la "U" o un tazón). Su gran ventaja es que solo tiene un punto mínimo, y ese mínimo es el mejor posible de todos (mínimo global).**

¿Por qué nos importa que una función sea convexa o no? ¿Qué impacto tiene para los resultados?

- *Nos importa porque define qué tan fácil o confiable es entrenar el modelo como:
Si es convexa, el algoritmo siempre llega al mejor resultado sin importar de dónde empiece.
Si es no convexa, el algoritmo se puede quedar atorado en mínimos locales (soluciones mediocres) o tardar mucho tiempo. Por eso los resultados dependen de la suerte con la que inicien los pesos y de ajustar bien los hiperparámetros.**

## Ejercicio 1

Utilizaremos el clasificador `MLPClassifier` de la librería `sklearn`.

Entra a la documentación y describe cómo funciona este objeto (qué entradas necesita, qué salidas da, descripción de sus hiperparámetros).

- **MLPClassifier es el perceptrón multicapa de scikit-learn.

Entradas: Matriz $X$ con variables numéricas escaladas y vector $y$ con las etiquetas binarias (0 o 1).
Salidas: Clases predichas (.predict()) y probabilidades por clase (.predict_proba()).
Hiperparámetros clave:
hidden_layer_sizes: Número de capas y neuronas por capa, como (64, 32).
activation: Función de activación ('relu', 'tanh', 'logistic').
solver: Optimizador para actualizar pesos ('adam', 'lbfgs', 'sgd').
learning_rate_init: Tasa de aprendizaje inicial.
max_iter: Número máximo de épocas de entrenamiento.
random_state: Semilla para reproducibilidad.**

## Ejercicio 2

Utiliza el objeto de perceptrón multicapa para clasificar si un tumor es maligno o benigno con el dataset `Brain Tumor Dataset`. Utiliza un cross-validation de train-test de 70/30. Sigue todos los pasos de EDA y limpieza vistos.

In [2]:
import pandas as pd

df = pd.read_csv("brain_tumor_dataset.csv")
print(df.head())
print(df.info())
print(df.isnull().sum())

df = df.drop(columns=["Patient_ID"])
y = df["Tumor_Type"].replace({"Benign": 0, "Malignant": 1}).astype(int)
print(y.value_counts())

X = df.drop(columns=["Tumor_Type"])
X = pd.get_dummies(X, drop_first=True)
print(X.head())
print(X.shape)

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)
print(X_train.shape, X_test.shape)

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
print(X_train)

from sklearn.neural_network import MLPClassifier
mlp = MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=200, random_state=42)
mlp.fit(X_train, y_train)

pred = mlp.predict(X_test)
proba = mlp.predict_proba(X_test)[:, 1]

from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, recall_score, f1_score, roc_auc_score
print(confusion_matrix(y_test, pred))
print(classification_report(y_test, pred))
print("Accuracy:", round(accuracy_score(y_test, pred), 4))
print("Recall:  ", round(recall_score(y_test, pred), 4))
print("F1-score:", round(f1_score(y_test, pred), 4))
print("ROC-AUC: ", round(roc_auc_score(y_test, proba), 4))


   Patient_ID  Age  Gender Tumor_Type  Tumor_Size  Location        Histology  \
0           1   73    Male  Malignant    5.375612  Temporal      Astrocytoma   
1           2   26    Male     Benign    4.847098  Parietal     Glioblastoma   
2           3   31    Male     Benign    5.588391  Parietal       Meningioma   
3           4   29    Male  Malignant    1.436600  Temporal  Medulloblastoma   
4           5   54  Female     Benign    2.417506  Parietal     Glioblastoma   

  Stage      Symptom_1 Symptom_2 Symptom_3 Radiation_Treatment  \
0   III  Vision Issues  Seizures  Seizures                  No   
1    II       Headache  Headache    Nausea                 Yes   
2     I  Vision Issues  Headache  Seizures                  No   
3    IV  Vision Issues  Seizures  Headache                 Yes   
4     I       Headache  Headache  Seizures                  No   

  Surgery_Performed Chemotherapy  Survival_Rate  Tumor_Growth_Rate  \
0                No           No      51.312579     

/opt/anaconda3/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


¿Qué métrico crees que sea mejor para este ejercicio? Escoge los 3 métricos que crees más apropiados para esta predicción, argumenta en pro y en contra del top 3. Elige uno para optimizar (justifica tu respuesta).

- **Top 3 métricos:**
1. **Recall:**
   * **Pro:** Se enfoca en no dejar escapar ningún tumor maligno (reduce los falsos negativos).
   * **Contra:** Puede dar muchas falsas alarmas con personas sanas.
2. **F1-Score:**
   * **Pro:** Busca un buen punto medio entre detectar enfermos y no dar falsas alarmas.
   * **Contra:** Trata los dos tipos de errores como si fueran igual de graves, y en salud no lo son.
3. **ROC-AUC:**
   * **Pro:** Mide qué tan bien separa los datos en general sin importar el punto de corte.
   * **Contra:** Es un número más teórico y no te da una respuesta clara de sí o no para un paciente.
---
### **Métrico que elijo optimizar: Recall**
**¿Por qué?**  
Porque estamos hablando de cáncer cerebral. Es mil veces peor decirle a alguien enfermo que está sano (porque puede morir si no se atiende) que asustar por error a alguien sano que luego se descarte con más estudios. Con Recall me aseguro de atrapar la mayor cantidad de tumores malignos posibles.

## Ejercicio 3

Modifica los hiperparámetros para elevar el métrico elegido. Intenta con 10 sets de hiperparámetros distintos. Crea una tabla que muestre los resultados.

In [3]:
import pandas as pd
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import recall_score, f1_score, roc_auc_score, accuracy_score

configs = [
    {"capas": (32,), "act": "relu", "lr": 0.001},
    {"capas": (64,), "act": "relu", "lr": 0.001},
    {"capas": (128,), "act": "relu", "lr": 0.001},
    {"capas": (64, 32), "act": "relu", "lr": 0.001},
    {"capas": (32, 16), "act": "relu", "lr": 0.001},
    {"capas": (32,), "act": "tanh", "lr": 0.001},
    {"capas": (64,), "act": "tanh", "lr": 0.001},
    {"capas": (32,), "act": "logistic", "lr": 0.005},
    {"capas": (64, 32), "act": "tanh", "lr": 0.001},
    {"capas": (64, 32), "act": "relu", "lr": 0.005},
]

resultados = []

for i, cfg in enumerate(configs, 1):
    modelo = MLPClassifier(hidden_layer_sizes=cfg["capas"], activation=cfg["act"], learning_rate_init=cfg["lr"], max_iter=100, random_state=42)
    modelo.fit(X_train, y_train)
    pred = modelo.predict(X_test)
    proba = modelo.predict_proba(X_test)[:, 1]
    
    r = recall_score(y_test, pred)
    f = f1_score(y_test, pred)
    auc = roc_auc_score(y_test, proba)
    acc = accuracy_score(y_test, pred)
    
    print(f"Set {i} -> Recall: {r:.4f} | F1: {f:.4f} | AUC: {auc:.4f} | Acc: {acc:.4f}")
    
    resultados.append({
        "Set": f"Set {i}",
        "Capas": str(cfg["capas"]),
        "Activacion": cfg["act"],
        "LR": cfg["lr"],
        "Recall": round(r, 4),
        "F1-Score": round(f, 4),
        "ROC-AUC": round(auc, 4),
        "Accuracy": round(acc, 4)
    })

tabla = pd.DataFrame(resultados).sort_values(by="Recall", ascending=False)
display(tabla)


/opt/anaconda3/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (100) reached and the optimization hasn't converged yet.
  warnings.warn(


Set 1 -> Recall: 0.5015 | F1: 0.5007 | AUC: 0.5043 | Acc: 0.4983


/opt/anaconda3/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (100) reached and the optimization hasn't converged yet.
  warnings.warn(


Set 2 -> Recall: 0.4616 | F1: 0.4784 | AUC: 0.4935 | Acc: 0.4952


/opt/anaconda3/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (100) reached and the optimization hasn't converged yet.
  warnings.warn(


...

Set 8 -> Recall: 0.5603 | F1: 0.5279 | AUC: 0.5018 | Acc: 0.4973


/opt/anaconda3/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (100) reached and the optimization hasn't converged yet.
  warnings.warn(


Set 9 -> Recall: 0.5171 | F1: 0.5134 | AUC: 0.5057 | Acc: 0.5085
Set 10 -> Recall: 0.4955 | F1: 0.5004 | AUC: 0.5079 | Acc: 0.5038


/opt/anaconda3/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (100) reached and the optimization hasn't converged yet.
  warnings.warn(


,Set,Capas,Activacion,LR,Recall,F1-Score,ROC-AUC,Accuracy
7,Set 8,"(32,)",logistic,0.005,0.5603,0.5279,0.5018,0.4973
4,Set 5,"(32, 16)",relu,0.001,0.5284,0.5095,0.4910,0.4898
3,Set 4,"(64, 32)",relu,0.001,0.5244,0.5128,0.5014,0.5002
8,Set 9,"(64, 32)",tanh,0.001,0.5171,0.5134,0.5057,0.5085
5,Set 6,"(32,)",tanh,0.001,0.5042,0.5040,0.4990,0.5023
0,Set 1,"(32,)",relu,0.001,0.5015,0.5007,0.5043,0.4983
9,Set 10,"(64, 32)",relu,0.005,0.4955,0.5004,0.5079,0.5038
2,Set 3,"(128,)",relu,0.001,0.4935,0.4983,0.4965,0.5017
6,Set 7,"(64,)",tanh,0.001,0.4826,0.4913,0.4992,0.4988
1,Set 2,"(64,)",relu,0.001,0.4616,0.4784,0.4935,0.4952


- **respuesta aquí**